# Resume ↔ Job Matcher
This notebook compares one resume to one job posting, explains the skills match, and gives truthful feedback on how to improve the resume.

In [18]:
from pathlib import Path
import re
import json
import sys
import random
from collections import defaultdict

BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / "Data"
RESUME_DIR = DATA_DIR / "Resumes"
JOB_DIR = DATA_DIR / "Job Postings"
PKG_DIR = BASE_DIR / "python_packages"

print("Notebook location:", BASE_DIR)
print("Resume folder:", RESUME_DIR)
print("Job folder:", JOB_DIR)

if str(PKG_DIR) not in sys.path and PKG_DIR.exists():
    sys.path.insert(0, str(PKG_DIR))

Notebook location: /home/csgrimberg/AI-Notebooks/IS Project
Resume folder: /home/csgrimberg/AI-Notebooks/IS Project/Data/Resumes
Job folder: /home/csgrimberg/AI-Notebooks/IS Project/Data/Job Postings


In [19]:
from docx import Document
import PyPDF2

print("Imports worked ✅")

Imports worked ✅


In [20]:
SUPPORTED_EXTENSIONS = {".txt", ".docx", ".pdf"}

def read_txt(path: Path) -> str:
    return path.read_text(encoding="utf-8", errors="ignore")

def read_docx(path: Path) -> str:
    doc = Document(path)
    return "\n".join(p.text for p in doc.paragraphs)

def read_pdf(path: Path) -> str:
    text = []
    with open(path, "rb") as f:
        reader = PyPDF2.PdfReader(f)
        for page in reader.pages:
            page_text = page.extract_text()
            if page_text:
                text.append(page_text)
    return "\n".join(text)

def read_file(path: Path) -> str:
    suffix = path.suffix.lower()
    if suffix == ".txt":
        return read_txt(path)
    if suffix == ".docx":
        return read_docx(path)
    if suffix == ".pdf":
        return read_pdf(path)
    raise ValueError(f"Unsupported file type: {suffix}")

def clean_text(text: str) -> str:
    text = text.replace("\xa0", " ")
    text = re.sub(r"\r", "\n", text)
    text = re.sub(r"\n{2,}", "\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    return text.strip()

In [21]:
def find_skill_file(base_dir: Path) -> Path:
    candidates = [
        base_dir / "Job Skills.txt",
        base_dir / "job skills.txt",
        base_dir / "skills.txt",
        base_dir / "Data" / "Job Skills.txt",
        base_dir / "Data" / "job skills.txt",
        base_dir / "Data" / "skills.txt",
    ]

    for candidate in candidates:
        if candidate.exists():
            return candidate

    txt_files = sorted(base_dir.rglob("*.txt"))
    for file in txt_files:
        if "skill" in file.name.lower():
            return file

    raise FileNotFoundError(
        "Could not find a skill mapping text file. Put 'Job Skills.txt' in the notebook folder or Data folder."
    )

SKILL_FILE = find_skill_file(BASE_DIR)
print("Using skill file:", SKILL_FILE)

Using skill file: /home/csgrimberg/AI-Notebooks/IS Project/Data/Job Skills.txt


In [22]:
def parse_skill_file(path: Path):
    """
    Expected format:
    [category_name]
    alias = Canonical Skill
    another alias = Canonical Skill
    """
    raw = read_txt(path)
    skill_aliases = {}
    canonical_to_category = {}
    current_category = "uncategorized"

    for line in raw.splitlines():
        line = line.strip()
        if not line or line.startswith("#"):
            continue

        if line.startswith("[") and line.endswith("]"):
            current_category = line[1:-1].strip()
            continue

        if "=" in line:
            alias, canonical = line.split("=", 1)
            alias = alias.strip().lower()
            canonical = canonical.strip()
            if alias:
                skill_aliases[alias] = canonical
                canonical_to_category[canonical] = current_category

    if not skill_aliases:
        raise ValueError("No skills were parsed from the skill file.")

    return skill_aliases, canonical_to_category

skill_aliases, canonical_to_category = parse_skill_file(SKILL_FILE)

print("Loaded aliases:", len(skill_aliases))
print("Loaded canonical skills:", len(set(skill_aliases.values())))

Loaded aliases: 264
Loaded canonical skills: 256


In [23]:
resume_files = sorted([f for f in RESUME_DIR.glob("*") if f.suffix.lower() in SUPPORTED_EXTENSIONS]) if RESUME_DIR.exists() else []
job_files = sorted([f for f in JOB_DIR.glob("*") if f.suffix.lower() in SUPPORTED_EXTENSIONS]) if JOB_DIR.exists() else []

print("Resumes found:", len(resume_files))
for f in resume_files:
    print("-", f.name)

print("\nJob Postings found:", len(job_files))
for f in job_files:
    print("-", f.name)

if not resume_files:
    raise FileNotFoundError(f"No resumes found in {RESUME_DIR}")
if not job_files:
    raise FileNotFoundError(f"No job postings found in {JOB_DIR}")

Resumes found: 4
- Catelin_Carnes_Resume.txt
- CeCe_Grimberg_Resume.txt
- Leo's Resume.txt
- LukesResume.txt

Job Postings found: 19
- Albea_Data_Analyst_Intern_Job_Description.txt
- BNP Parabis.txt
- Bank of America CAPEM.txt
- Bank of America Subscription.txt
- Big_Happy_Data_Engineering_Intern_Job_Description.txt
- Blackstone LaunchPad Internship.txt
- CareFirst_Data_Analytics_Intern_Job_Description.txt
- DP Solutions Internship.txt
- Film Forum Internship.txt
- Job Postings.docx
- Lockheed Martin Financial Analyst.txt
- Madison Energies M&A.txt
- MarketSync Media Internship.txt
- Proactive_Worldwide_Job_Description.txt
- Scotiabank IB intern.txt
- US_World_Alliance_Industrial_Data_Analyst_Job_Description.txt
- Under Armour Internship.txt
- Vatn_Systems_Data_Analyst_Intern_Job_Description.txt
- Vice Versa Media Internship.txt


In [24]:
def alias_in_text(alias: str, text_lower: str) -> bool:
    alias = alias.lower().strip()
    if not alias:
        return False

    # handle special cases like c++, c#, node.js, a/b testing
    if re.search(r"[+#./&-]", alias):
        pattern = re.escape(alias)
        return re.search(pattern, text_lower) is not None

    pattern = r"\b" + re.escape(alias) + r"\b"
    return re.search(pattern, text_lower) is not None

def extract_skills(text: str, skill_aliases: dict) -> list:
    text_lower = text.lower()
    found = set()

    for alias, canonical in skill_aliases.items():
        if alias_in_text(alias, text_lower):
            found.add(canonical)

    return sorted(found)

def group_skills_by_category(skills, canonical_to_category):
    grouped = defaultdict(list)
    for skill in skills:
        grouped[canonical_to_category.get(skill, "uncategorized")].append(skill)
    return {k: sorted(v) for k, v in sorted(grouped.items())}

In [25]:
def compare_skills(resume_skills: list, job_skills: list) -> dict:
    resume_set = set(resume_skills)
    job_set = set(job_skills)

    matched = sorted(resume_set & job_set)
    missing = sorted(job_set - resume_set)
    extra_resume_skills = sorted(resume_set - job_set)

    score = round((len(matched) / len(job_set)) * 100, 2) if job_set else 0.0

    return {
        "matched": matched,
        "missing": missing,
        "extra_resume_skills": extra_resume_skills,
        "match_percent": score
    }

def classify_match(score: float) -> str:
    if score >= 75:
        return "Strong alignment"
    if score >= 50:
        return "Moderate alignment"
    if score >= 30:
        return "Partial alignment"
    return "Low alignment"

In [26]:
def generate_skill_specific_idea(skill: str, category: str = "uncategorized") -> str:
    skill_lower = skill.lower()

    direct_templates = {
        "sql": "Add a bullet that shows how you used SQL to query, join, clean, or summarize data, and include what decision or result it supported.",
        "python": "Add a bullet that explains how you used Python for analysis, automation, scraping, modeling, or reporting, with a measurable output.",
        "excel": "Show Excel through a concrete task like pivot tables, lookups, modeling, dashboards, or reporting, and tie it to time saved or insight delivered.",
        "tableau": "Mention a Tableau dashboard or visualization you built, what audience used it, and what business question it helped answer.",
        "power bi": "Add a Power BI example that highlights dashboard creation, reporting automation, or KPI tracking for a team or project.",
        "communication": "Turn communication into proof by adding a bullet about presenting findings, writing client-facing materials, or coordinating across teams.",
        "presentation skills": "Add a line showing when you presented recommendations, project updates, or analysis to a class, client, or leadership group.",
        "writing": "Show writing through reports, proposals, summaries, briefs, or documentation that had a clear audience and purpose.",
        "leadership": "Use a bullet that shows initiative, ownership, mentoring, or leading part of a team project with a concrete outcome.",
        "project management": "Show project management with planning, deadlines, coordination, and deliverables rather than listing it as a standalone skill.",
        "agile": "Reference a project where you worked in sprints, tracked tasks, or iterated based on feedback to show Agile in practice.",
        "jira": "Mention using Jira to track tickets, manage workflows, or coordinate tasks across a project team.",
        "machine learning": "Add a project bullet that explains the dataset, model approach, evaluation method, and the result you achieved.",
        "artificial intelligence": "Tie AI to a real project, tool, or workflow improvement so it sounds applied rather than generic.",
        "data analysis": "Add a bullet that explains what data you analyzed, what method you used, and what insight or recommendation came from it.",
        "data visualization": "Show how you turned data into charts, dashboards, or visual stories that improved understanding for others.",
        "financial analysis": "Add a bullet describing the model, metrics, or trend analysis you performed and what recommendation or conclusion it supported.",
        "financial modeling": "Show the type of model you built, the inputs you analyzed, and how the model informed planning or valuation.",
        "marketing": "Add a bullet with the campaign, audience, channel, and performance result so marketing experience feels specific and measurable.",
        "sales": "Show sales with outreach volume, conversion rate, pipeline support, account growth, or event-based revenue.",
        "crm": "Mention the CRM platform, what data you managed, and how it improved outreach, reporting, or customer tracking.",
        "lead generation": "Describe how you sourced leads, qualified prospects, or improved pipeline development with a metric when possible.",
        "operations": "Add a bullet that shows how you improved workflow efficiency, coordination, accuracy, or turnaround time.",
        "supply chain": "Show supply chain work through forecasting, logistics, inventory, sourcing, or process optimization examples.",
        "research": "Add a bullet about the research question, sources used, and how your findings shaped a recommendation or decision.",
        "strategy": "Show strategy through market analysis, benchmarking, recommendations, or planning decisions tied to a real project.",
        "ux": "Show UX with research, wireframes, testing, or design changes you made based on user needs.",
        "ui": "Mention interface mockups, design systems, or visual changes you created and why they improved usability.",
        "apis": "Mention connecting to an API, pulling data, or building an integration and explain what that enabled.",
        "rest apis": "Show REST API knowledge through an integration, request workflow, or backend/frontend project example.",
        "testing": "Add a bullet explaining what kind of testing you performed and how it improved reliability or reduced errors.",
        "debugging": "Describe a time you identified and fixed a bug, issue, or bottleneck and what impact the fix had.",
    }

    category_templates = {
        "data_analytics": "Add a bullet showing the dataset, tool, analysis method, and the insight or metric that came out of your work with {skill}.",
        "machine_learning_ai": "Add a project line for {skill} that covers the problem, approach, tools used, and the model or workflow outcome.",
        "software_engineering": "Show {skill} through a build, feature, integration, or debugging example that makes the skill feel demonstrated, not claimed.",
        "cloud_data_engineering": "Mention how you used {skill} in a pipeline, cloud environment, warehouse, or deployment workflow and what it improved.",
        "finance_accounting": "Add a bullet showing how {skill} supported budgeting, modeling, reporting, valuation, or decision-making.",
        "marketing_sales": "Show {skill} with a campaign, outreach effort, audience target, funnel metric, or client-facing result.",
        "operations_supply_chain": "Use a bullet where {skill} is tied to workflow improvement, planning, logistics, inventory, or operational efficiency.",
        "consulting_strategy": "Show {skill} with research, benchmarking, recommendation development, or a client/team decision it informed.",
        "project_management": "Describe how you used {skill} to plan tasks, manage deadlines, coordinate people, or keep a deliverable on track.",
        "communication": "Turn {skill} into evidence by connecting it to a report, presentation, stakeholder update, or documented recommendation.",
        "design_media": "Show {skill} through a design, prototype, content asset, or creative deliverable and explain its purpose.",
        "human_resources": "Mention how {skill} supported recruiting, onboarding, training, communication, or employee support.",
        "healthcare": "Add a line showing how {skill} connected to patient support, compliance, records, research, or healthcare operations.",
        "office_tools": "Instead of only listing {skill}, pair it with a task like reporting, coordination, documentation, scheduling, or presentation work.",
        "uncategorized": "If you have real experience with {skill}, add a bullet that explains where you used it, what you did, and what outcome followed."
    }

    if skill_lower in direct_templates:
        return direct_templates[skill_lower]

    template = category_templates.get(category, category_templates["uncategorized"])
    return template.format(skill=skill)

def generate_resume_feedback(resume_text: str, job_text: str, matched_skills: list, missing_skills: list, canonical_to_category: dict):
    feedback = {
        "recommended_keywords": [],
        "rewording_suggestions": [],
        "skill_improvement_ideas": {},
        "section_suggestions": [],
        "priority_skills": missing_skills[:10],
        "summary": ""
    }

    for skill in missing_skills:
        category = canonical_to_category.get(skill, "uncategorized")
        feedback["recommended_keywords"].append(skill)

        idea = generate_skill_specific_idea(skill, category)
        feedback["rewording_suggestions"].append(f"{skill}: {idea}")
        feedback["skill_improvement_ideas"][skill] = {
            "category": category,
            "idea": idea
        }

    if missing_skills:
        feedback["section_suggestions"].append(
            "Add a targeted 'Relevant Skills' section near the top so the most job-relevant tools and capabilities appear earlier."
        )
        feedback["section_suggestions"].append(
            "Revise 2–4 experience bullets so they include the tool used, the task performed, and the result or outcome."
        )
        feedback["section_suggestions"].append(
            "Mirror the job posting language when it truthfully matches your actual coursework, projects, internships, or work experience."
        )

    if matched_skills:
        feedback["section_suggestions"].append(
            "Move your strongest matched skills into the summary, top skills block, or most recent experience section so they are seen faster."
        )

    total_job_skills = len(matched_skills) + len(missing_skills)
    feedback["summary"] = (
        f"The resume currently matches {len(matched_skills)} of {total_job_skills} extracted job skills. "
        "To improve your chances, keep the matched skills visible near the top and add missing skills only when you can back them up truthfully with experience."
    )

    for key in ["recommended_keywords", "rewording_suggestions", "section_suggestions"]:
        seen = set()
        deduped = []
        for item in feedback[key]:
            if item not in seen:
                deduped.append(item)
                seen.add(item)
        feedback[key] = deduped

    return feedback

def estimate_potential_match(current_match: float, missing_skills: list) -> float:
    if not missing_skills:
        return current_match

    possible_gain = min(len(missing_skills) * 4, 30)
    potential = min(current_match + possible_gain, 95.0)
    return round(potential, 2)


In [27]:
# OPTION 1: choose files by name
RESUME_NAME = None   # example: "CeCe_Resume.docx"
JOB_NAME = None      # example: "Data Analyst Job.txt"

def select_file(files, chosen_name=None):
    if chosen_name:
        for f in files:
            if f.name == chosen_name:
                return f
        raise FileNotFoundError(f"Could not find file named: {chosen_name}")
    return random.choice(files)

sample_resume = select_file(resume_files, RESUME_NAME)
sample_job = select_file(job_files, JOB_NAME)

print("Selected resume:", sample_resume.name)
print("Selected job posting:", sample_job.name)

Selected resume: Catelin_Carnes_Resume.txt
Selected job posting: Bank of America Subscription.txt


In [28]:
resume_text = clean_text(read_file(sample_resume))
job_text = clean_text(read_file(sample_job))

resume_skills = extract_skills(resume_text, skill_aliases)
job_skills = extract_skills(job_text, skill_aliases)

results = compare_skills(resume_skills, job_skills)
feedback = generate_resume_feedback(
    resume_text=resume_text,
    job_text=job_text,
    matched_skills=results["matched"],
    missing_skills=results["missing"],
    canonical_to_category=canonical_to_category
)
potential_match = estimate_potential_match(
    current_match=results["match_percent"],
    missing_skills=results["missing"]
)

matched_by_category = group_skills_by_category(results["matched"], canonical_to_category)
missing_by_category = group_skills_by_category(results["missing"], canonical_to_category)
extra_by_category = group_skills_by_category(results["extra_resume_skills"], canonical_to_category)


In [29]:
print("=" * 70)
print("RESUME vs JOB MATCH REVIEW")
print("=" * 70)
print("Resume:", sample_resume.name)
print("Job Posting:", sample_job.name)
print("\nCurrent Match %:", results["match_percent"])
print("Potential Match % After Truthful Resume Updates:", potential_match)
print("Overall Read:", classify_match(results["match_percent"]))

print("\nSUMMARY")
print(feedback["summary"])

print("\nMATCHED SKILLS")
if results["matched"]:
    for skill in results["matched"]:
        print("-", skill)
else:
    print("- None found")

print("\nMISSING JOB SKILLS")
if results["missing"]:
    for skill in results["missing"]:
        print("-", skill)
else:
    print("- None found")

print("\nEXTRA SKILLS ON RESUME (not emphasized by this posting)")
if results["extra_resume_skills"]:
    for skill in results["extra_resume_skills"][:20]:
        print("-", skill)
else:
    print("- None found")

RESUME vs JOB MATCH REVIEW
Resume: Catelin_Carnes_Resume.txt
Job Posting: Bank of America Subscription.txt

Current Match %: 16.67
Potential Match % After Truthful Resume Updates: 36.67
Overall Read: Low alignment

SUMMARY
The resume currently matches 1 of 6 extracted job skills. To improve your chances, keep the matched skills visible near the top and add missing skills only when you can back them up truthfully with experience.

MATCHED SKILLS
- Statistics

MISSING JOB SKILLS
- Cash Flow Analysis
- Finance
- Private Equity
- Sales
- Training & Development

EXTRA SKILLS ON RESUME (not emphasized by this posting)
- Adobe Premiere Pro
- Artificial Intelligence
- Data Visualization
- Excel
- Operations
- Problem Solving
- Python
- R
- SAS
- SQL
- Tableau


In [30]:
print("\nSKILL MATCH BY CATEGORY")
for category, skills in matched_by_category.items():
    print(f"\n[{category}]")
    for skill in skills:
        print("-", skill)

print("\nMISSING SKILLS BY CATEGORY")
for category, skills in missing_by_category.items():
    print(f"\n[{category}]")
    for skill in skills:
        print("-", skill)


SKILL MATCH BY CATEGORY

[data_analytics]
- Statistics

MISSING SKILLS BY CATEGORY

[finance_accounting]
- Cash Flow Analysis
- Finance
- Private Equity

[human_resources]
- Training & Development

[marketing_sales]
- Sales


In [31]:
print("\nHOW TO IMPROVE THE RESUME")
for item in feedback["section_suggestions"]:
    print("-", item)

print("\nKEYWORDS TO ADD OR BETTER HIGHLIGHT")
for item in feedback["recommended_keywords"][:15]:
    print("-", item)

print("\nUNIQUE REWORDING IDEAS BY MISSING SKILL")
for skill, detail in list(feedback["skill_improvement_ideas"].items())[:15]:
    print(f"- {skill} [{detail['category']}]: {detail['idea']}")



HOW TO IMPROVE THE RESUME
- Add a targeted 'Relevant Skills' section near the top so the most job-relevant tools and capabilities appear earlier.
- Revise 2–4 experience bullets so they include the tool used, the task performed, and the result or outcome.
- Mirror the job posting language when it truthfully matches your actual coursework, projects, internships, or work experience.
- Move your strongest matched skills into the summary, top skills block, or most recent experience section so they are seen faster.

KEYWORDS TO ADD OR BETTER HIGHLIGHT
- Cash Flow Analysis
- Finance
- Private Equity
- Sales
- Training & Development

UNIQUE REWORDING IDEAS BY MISSING SKILL
- Cash Flow Analysis [finance_accounting]: Add a bullet showing how Cash Flow Analysis supported budgeting, modeling, reporting, valuation, or decision-making.
- Finance [finance_accounting]: Add a bullet showing how Finance supported budgeting, modeling, reporting, valuation, or decision-making.
- Private Equity [finance_

In [32]:
print("\nSAMPLE BULLET IDEAS")
for skill, detail in list(feedback["skill_improvement_ideas"].items())[:5]:
    print(f"- {skill}: {detail['idea']}")



SAMPLE BULLET IDEAS
- Cash Flow Analysis: Add a bullet showing how Cash Flow Analysis supported budgeting, modeling, reporting, valuation, or decision-making.
- Finance: Add a bullet showing how Finance supported budgeting, modeling, reporting, valuation, or decision-making.
- Private Equity: Add a bullet showing how Private Equity supported budgeting, modeling, reporting, valuation, or decision-making.
- Sales: Show sales with outreach volume, conversion rate, pipeline support, account growth, or event-based revenue.
- Training & Development: Mention how Training & Development supported recruiting, onboarding, training, communication, or employee support.


In [33]:
output = {
    "resume_file": sample_resume.name,
    "job_file": sample_job.name,
    "resume_skills": resume_skills,
    "job_skills": job_skills,
    "current_match_percent": results["match_percent"],
    "potential_match_percent": potential_match,
    "alignment_label": classify_match(results["match_percent"]),
    "matched_skills": results["matched"],
    "missing_skills": results["missing"],
    "extra_resume_skills": results["extra_resume_skills"],
    "matched_by_category": matched_by_category,
    "missing_by_category": missing_by_category,
    "recommended_keywords": feedback["recommended_keywords"],
    "rewording_suggestions": feedback["rewording_suggestions"],
    "skill_improvement_ideas": feedback["skill_improvement_ideas"],
    "section_suggestions": feedback["section_suggestions"],
    "summary": feedback["summary"]
}

output_path = BASE_DIR / "resume_job_match_result.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(output, f, indent=2)

print("Saved results to:", output_path)


Saved results to: /home/csgrimberg/AI-Notebooks/IS Project/resume_job_match_result.json


In [39]:
import os
import random

from app.skills_loader import load_skill_dictionary
from app.parsers import read_text_file, normalize_text
from app.matcher import extract_skills, compare_resume_to_job
from app.feedback import generate_rule_based_feedback

skills = load_skill_dictionary("data/job_skills.txt")

resume_files = os.listdir("data/resumes")
job_files = os.listdir("data/job_postings")

resume_path = f"data/resumes/{random.choice(resume_files)}"
job_path = f"data/job_postings/{random.choice(job_files)}"

print("Using resume:", resume_path)
print("Using job:", job_path)

resume_text = normalize_text(read_text_file(resume_path))
job_text = normalize_text(read_text_file(job_path))

resume_skills = extract_skills(resume_text, skills)
job_skills = extract_skills(job_text, skills)

match_result = compare_resume_to_job(resume_skills, job_skills)
feedback = generate_rule_based_feedback(match_result)

print("\nSCORE:", match_result["score"])
print("\nMATCHED:", match_result["matched_skills"])
print("\nMISSING:", match_result["missing_skills"])
print("\nFEEDBACK:", feedback)

Using resume: data/resumes/CeCe_Grimberg_Resume.txt
Using job: data/job_postings/DP Solutions Internship.txt

SCORE: 14.3

MATCHED: ['Marketing', 'Operations']

MISSING: ['B2B Sales', 'Communication', 'Digital Marketing', 'Editing', 'Graphic Design', 'Logistics', 'Market Research', 'Planning', 'Sales', 'Social Media Marketing', 'Strategy', 'Writing']

FEEDBACK: {'fit_summary': 'Weaker fit. The resume needs clearer alignment with the job posting.', 'strongest_matches': ['You already show evidence of Marketing.', 'You already show evidence of Operations.'], 'biggest_gaps': ['The job asks for B2B Sales, but it is not clearly shown in the resume.', 'The job asks for Communication, but it is not clearly shown in the resume.', 'The job asks for Digital Marketing, but it is not clearly shown in the resume.', 'The job asks for Editing, but it is not clearly shown in the resume.', 'The job asks for Graphic Design, but it is not clearly shown in the resume.'], 'resume_improvements': ['Add truthf

In [40]:
from app.analyzer import analyze_resume_text_against_job_text

resume_text = "Python SQL Excel data analysis dashboard reporting teamwork communication"
job_text = "Looking for Python SQL Power BI dashboard development communication"

result = analyze_resume_text_against_job_text(resume_text, job_text)

print(result["match_result"])
print(result["feedback"])

{'score': 80.0, 'matched_skills': ['Communication', 'Dashboard Development', 'Python', 'SQL'], 'missing_skills': ['Power BI'], 'category_breakdown': {'communication': {'matched': ['Communication'], 'missing': [], 'resume_only': []}, 'data_analytics': {'matched': ['Dashboard Development', 'Python', 'SQL'], 'missing': ['Power BI'], 'resume_only': ['Excel', 'Reporting']}, 'office_tools': {'matched': [], 'missing': [], 'resume_only': ['Excel']}}}
{'fit_summary': "Strong fit. The resume aligns well with the job's requested skills.", 'strongest_matches': ['You already show evidence of Communication.', 'You already show evidence of Dashboard Development.', 'You already show evidence of Python.', 'You already show evidence of SQL.'], 'biggest_gaps': ['The job asks for Power BI, but it is not clearly shown in the resume.'], 'resume_improvements': ['Add truthful examples that show Power BI through coursework, projects, internships, or volunteer work.']}


In [41]:
!pip install fastapi uvicorn python-multipart

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.7/117.7 kB 4.1 MB/s eta 0:00:00
  Attempting uninstall: typing-inspection
    Found existing installation: typing-inspection 0.4.1
    Uninstalling typing-inspection-0.4.1:
ERROR: Could not install packages due to an OSError: [Errno 13] Permission denied: 'METADATA'
Check the permissions.



In [44]:
uvicorn app.main:app

SyntaxError: invalid syntax (172609951.py, line 1)